# Lab6 

Bryan Edgardo Romo Gonzalez
---
Luis Fernando Del Real Vazquez
---
Omar Oswaldo Ramirez Arenas
---

In [1]:
!pip install graphframes

In [ ]:
from spark_utils import SparkUtils
neo4j_connector = "org.neo4j:neo4j-connector-apache-spark_2.13:5.3.10_for_spark_3,io.graphframes:graphframes-spark3_2.13:0.9.0-spark3.5"
su = SparkUtils("Lab 6", "spark://spark-master:7077", spark_packages=neo4j_connector)
su._spark

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.neo4j#neo4j-connector-apache-spark_2.13 added as a dependency
io.graphframes#graphframes-spark3_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-eabe4e2f-b008-43ae-b404-8bee2a6654e8;1.0
	confs: [default]
	found org.neo4j#neo4j-connector-apache-spark_2.13;5.3.10_for_spark_3 in central
	found org.neo4j#neo4j-connector-apache-spark_2.13_common;5.3.10_for_spark_3 in central
	found org.neo4j#caniuse-core;1.3.0 in central
	found org.neo4j#caniuse-api;1.3.0 in central
	found org.jetbrains.kotlin#kotlin-stdlib;2.1.20 in central
	found org.jetbrains#annotations;13.0 in central
	found org.neo4j#caniuse-neo4j-detection;1.3.0 in central
	found org.neo4j.driver#neo4j-java-driver-slim;4.4.21 in central
	found org.reactivestreams#reactiv


## Data Ingestion

In [3]:

videogames_schema = SparkUtils.generate_schema([("user", "string"), ("videogame", "string"), ("rating", "float"), ("timestamp", "int")])


videogames_df = su._spark \
                .read \
                .option("header", "false") \
                .option("sep", ",") \
                .schema(videogames_schema) \
                .csv("/opt/spark/work-dir/data/videogames/")


videogames_df.show()

[Stage 0:>                                                          (0 + 1) / 1]

+--------------+----------+------+----------+
|          user| videogame|rating| timestamp|
+--------------+----------+------+----------+
| AB9S9279OZ3QO|0078764343|   5.0|1373155200|
|A24SSUT5CSW8BH|0078764343|   5.0|1377302400|
| AK3V0HEBJMQ7J|0078764343|   4.0|1372896000|
|A10BECPH7W8HM7|043933702X|   5.0|1404950400|
|A2PRV9OULX1TWP|043933702X|   5.0|1386115200|
| AE7GUHCDQQ4UI|043933702X|   1.0|1366156800|
| A48ABFDDRMKI8|043933702X|   5.0|1374192000|
|A26B0P6K95SIKW|0439339960|   3.0|1288569600|
| AZ3UWOC8QSO6C|0439339987|   5.0|1366848000|
|A182S3ANC0W7DL|0439342260|   4.0|1355875200|
| AY5Q951JAZAX9|0439374391|   5.0|1356393600|
|A1TL721YECDIM8|0439394422|   3.0|1389657600|
| APDCEJMFDO2YT|0439394422|   5.0|1285545600|
| AFJ7A9CSEPZNY|043940133X|   4.0|1168300800|
|A14YVGE643TRJK|043940133X|   1.0|1324944000|
|A2H3TQWU51W1WE|043940133X|   5.0|1299456000|
|A3KO10N2ODLHBR|043940133X|   5.0|1258329600|
|A1T98OCCYW6OBI|043940133X|   2.0|1343779200|
|A3P758JGM12DD7|043940133X|   5.0|

# Graph Analysis

## Build GraphFrame

In [4]:
from graphframes import GraphFrame
from pyspark.sql.functions import col

users = videogames_df.select(col("user").alias("id")).distinct()
products = videogames_df.select(col("videogame").alias("id")).distinct()
vertices = users.union(products).distinct()

edges = videogames_df.select(
    col("user").alias("src"),
    col("videogame").alias("dst"),
    col("rating")
)

g = GraphFrame(vertices, edges)
print(f"Vertices: {g.vertices.count()}")
print(f"Edges: {g.edges.count()}")

/opt/spark/python/pyspark/sql/classic/dataframe.py:146: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(
                                                                                

Vertices: 876977


[Stage 15:=============================>                            (1 + 1) / 2]

Edges: 1324753


## PageRank

In [28]:
results = g.pageRank(resetProbability=0.15, maxIter=10)

results.vertices \
    .select("id", "pagerank") \
    .orderBy("pagerank", ascending=False) \
    .show()

[Stage 1215:============================================>           (4 + 1) / 5]

+----------+------------------+
|        id|          pagerank|
+----------+------------------+
|B00DJFIMW6| 7285.526431395484|
|B00BGA9WK2| 2645.507195083908|
|B00FAX6XQC|2547.3341328377564|
|B009KS4XRO|2501.0385889444806|
|B0055SWM08|1989.4563258041183|
|B00CSR2J9I| 1964.373283273377|
|B002VBWIP6|1857.1934441848766|
|B0015AARJI|1408.3694893308366|
|B000FKBCX4|1243.5554664378442|
|B00178630A| 1230.274656615952|
|B007VTVRFA|1206.0855893139276|
|B00AIUUXHC| 1173.889544941537|
|B003VANOFY| 986.3206066999797|
|B007FTE2VW| 984.3404153958518|
|B000B9RI14| 933.3473057681464|
|B00EF1OGOG|  813.309107754935|
|B005EQE0YM| 789.8962020979171|
|B0009VXBAQ| 759.9537257156875|
|B0029LJIFG| 752.2411136270933|
|B00AW3FDIO| 740.7225573743946|
+----------+------------------+
only showing top 20 rows


## Label Propagation

In [29]:
lpa = g.labelPropagation(maxIter=5)
lpa.show()

[Stage 1444:============================================>           (4 + 1) / 5]

+--------------------+-----------+
|                  id|      label|
+--------------------+-----------+
|          043933702X|25769804420|
|          0439671418| 8589972610|
|          0439900581|17179871748|
|          0545115507|      55201|
|          0700026657| 8590005570|
|          1886846758|17179911574|
|          7293000936| 8590098049|
|          7542614444|25769958148|
|          9078439122|34359872010|
|          9861064222|17179908478|
|          986118452X| 8590070871|
|          9861767304| 8589942402|
|          986325083X|25769883582|
|          9941113300|      94401|
|A0009878M2RGMMHGJH39|     169495|
|A00101961G0VS92WD...| 8590100259|
|A001147626R4BL248...|25769977765|
|A0011756FPL8K71Q5TAQ|34359907769|
|A00230923E4Y7VHWZ...| 8590107813|
|A00338543M2OZPUWO...|     172460|
+--------------------+-----------+
only showing top 20 rows


## Triangle Counting

In [30]:
triangle_count = g.triangleCount()
triangle_count.show()

[Stage 1581:============================================>           (4 + 1) / 5]

+-----+--------------+
|count|            id|
+-----+--------------+
|    0|A24SSUT5CSW8BH|
|    0| AE7GUHCDQQ4UI|
|    0| AY5Q951JAZAX9|
|    0| APDCEJMFDO2YT|
|    0|A14YVGE643TRJK|
|    0|A1TBUSGCBTXWFC|
|    0|A221V1CP1DVGRI|
|    0|A129SW886TYQ6H|
|    0| AFWPLXT2OD6H1|
|    0|A34UGG0WDKRTQF|
|    0|A1WJW2OIR5BZ58|
|    0|A1VJVU8YSBPAY7|
|    0|A11DR0TXTI0P0L|
|    0|A1JPYSVXO2PV36|
|    0|A1EO9BFUHTGWKZ|
|    0| A6Q1IUFSN2E2D|
|    0|A1LMJ9W8UX1H5B|
|    0|A3GB9OF65GKZF7|
|    0|A1125BZR4DT8RR|
|    0|A1NCF8YOK4YVY8|
+-----+--------------+
only showing top 20 rows


## Degree Distribution
### InDegree

In [31]:
in_deg = g.inDegrees.join(vertices, "id")
in_deg.orderBy("inDegree", ascending=False).show()

[Stage 1597:=====================================>                  (2 + 1) / 3]

+----------+--------+
|        id|inDegree|
+----------+--------+
|B00DJFIMW6|   16221|
|B00BGA9WK2|    7561|
|B00FAX6XQC|    5713|
|B009KS4XRO|    5489|
|B002VBWIP6|    5190|
|B0055SWM08|    4638|
|B00CSR2J9I|    4510|
|B0015AARJI|    4468|
|B00178630A|    3522|
|B000FKBCX4|    3290|
|B007VTVRFA|    3122|
|B000B9RI14|    2996|
|B007FTE2VW|    2743|
|B00AIUUXHC|    2739|
|B003VANOFY|    2572|
|B0009VXBAQ|    2409|
|B0050SYX8W|    2346|
|B0029LJIFG|    2309|
|B00CMQTVK0|    2219|
|B005EQE0YM|    2194|
+----------+--------+
only showing top 20 rows


### OutDegree

In [32]:
out_deg = g.outDegrees.join(vertices, "id")
out_deg.orderBy("outDegree", ascending=False).show()

[Stage 1608:=====================================>                  (2 + 1) / 3]

+--------------+---------+
|            id|outDegree|
+--------------+---------+
|A3V6Z4RCDGRC44|      880|
|A3W4D8XOGLWUN5|      817|
| AJKWF4W7QD4NS|      797|
|A2QHS1ZCIQOL7E|      521|
|A2TCG2HV1VJP6V|      474|
|A29BQ6B90Y1R5F|      429|
| AFV2584U13XP3|      338|
|A20DZX38KRBIT8|      320|
| A74TA8X5YQ7NE|      267|
|A2582KMXLK2P06|      263|
|A3GKMQFL05Z79K|      253|
|A3J8ABVGK7ZL6H|      250|
| AQMWZIH22R6LE|      239|
|A1AISPOIIHTHXX|      227|
|A1LBAC84TLIGAX|      224|
|A2GBBDNZLYC4A9|      222|
| A8NHN9UPML858|      217|
| AWG2O9C42XW5G|      212|
| ANAYSRE3LX8GZ|      210|
|A319SKSB556033|      209|
+--------------+---------+
only showing top 20 rows


# Writing Data in Neo4j

In [5]:
neo4j_url = "bolt://neo4j-iteso:7687"
neo4j_user = "neo4j"
neo4j_passwd = "neo4j@1234"

In [7]:

neo4j_options = {
    "url": neo4j_url, 
    "authentication.basic.username": neo4j_user, 
    "authentication.basic.password": neo4j_passwd, 
    "batch.size": "10000",
    "transaction.retries": "3"
}

In [8]:
from pyspark.sql.functions import col



user_vertices = videogames_df.select(col("user").alias("id")).distinct() 

user_vertices.write \
    .format("org.neo4j.spark.DataSource") \
    .mode("Overwrite") \
    .options(**neo4j_options) \
    .option("labels", "User") \
    .option("node.keys", "id") \
    .save()

print(f"{user_vertices.count()} user vertices wrote in Neo4j")

[Stage 23:======================================>                   (2 + 1) / 3]

826767 user vertices wrote in Neo4j


In [9]:
product_vertices = videogames_df.select(col("videogame").alias("id")).distinct() 

product_vertices.write \
    .format("org.neo4j.spark.DataSource") \
    .mode("Overwrite") \
    .options(**neo4j_options) \
    .option("labels", "Videogame") \
    .option("node.keys", "id") \
    .save()

print(f"{product_vertices.count()} videogame vertices wrote in Neo4j")

[Stage 30:=============================>                            (1 + 1) / 2]

50210 videogame vertices wrote in Neo4j


In [10]:
rating_edges = videogames_df.select(
    col("user").alias("src_id"),
    col("videogame").alias("dst_id"),
    col("rating"),
    col("timestamp")
) 
rel_query = """
MATCH (u:User {id: event.src_id})
MATCH (v:Videogame {id: event.dst_id})
MERGE (u)-[r:RATED]->(v)
SET r.rating = event.rating, 
    r.timestamp = event.timestamp
"""

rating_edges.write \
    .format("org.neo4j.spark.DataSource") \
    .mode("Append") \
    .options(**neo4j_options) \
    .option("query", rel_query) \
    .save()

print(f"{rating_edges.count()} edges wrote in Neo4j") 
su.spark.stop() 

1324753 edges wrote in Neo4j


In [11]:
su.spark.stop()